In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.read.format('parquet')\
              .option('header',True)\
              .option('inferSchema',True)\
              .load('abfss://bronze@xc797demo.dfs.core.windows.net/athletes').display()

In [0]:
df = spark.read.format('parquet')\
              .option('header',True)\
              .option('inferSchema',True)\
              .load('abfss://bronze@xc797demo.dfs.core.windows.net/athletes')

In [0]:
df.limit(5).display()

In [0]:
df = df.fillna("unknown", subset=["birth_place", "birth_country", "residence_place", "residence_country"])

In [0]:
df.display()

In [0]:
# df = df.fillna({"birth_place": "xyz", "birth_country": "abc", "residence_place": "unknown", "residence_country": "aaa"})

In [0]:
df.groupBy("current").count().display()

In [0]:
# df_filter = df.filter((col('current') == 'True') & (col('name').isin ('GALSTYAN Slavik', 'HARUTYUNYAN Arsen', 'SEHEN Sajjad')))
# df_filter.limit(5).display()


In [0]:
from pyspark.sql.functions import col

# Filter for active athletes with specific names
df_filter = df.filter(
    (col('current') == True) &
    (col('name').isin(['GALSTYAN Slavik', 'HARUTYUNYAN Arsen', 'SEHEN Sajjad']))
)

# Verify how many rows were matched
print(f"Matched rows: {df_filter.count()}")

# Preview the filtered data
df_filter.limit(5).display()

In [0]:
df.limit(5).display()

In [0]:
df.printSchema()

In [0]:
# schema issue
df = df.withColumn('height', col('height').cast('float'))\
    .withColumn('weight', col('weight').cast('float'))
df.display()

In [0]:
df_sort = df.sort('height', 'weight', ascending=[0,1]).filter(col('weight')>0)
df_sort.display()

In [0]:
df_sort = df_sort.withColumn('nationality', regexp_replace('nationality','United States', 'US'))
df_sort.display()

In [0]:
# no duplicate 
df.groupBy("code").agg(count("code").alias("total_count")).filter(col("total_count") > 1).display()

In [0]:
df_sort = df_sort.withColumnRenamed('code', 'athlete_code')
df_sort.display()

In [0]:
df_sort = df_sort.withColumn('occupation', split(col('occupation'), ','))
df_sort.display()

In [0]:
from pyspark.sql.functions import split, trim, transform

df_sort = df_sort.withColumn(
    'occupation',
    transform(split(col('occupation'), ','), lambda x: trim(x))
)

In [0]:
df_sort.display()

In [0]:
df_sort.columns

In [0]:
df_final = df_sort.select('athlete_code',
 'current',
 'name',
 'name_short',
 'name_tv',
 'gender',
 'function',
 'country_code',
 'country',
 'country_long',
 'nationality',
 'nationality_long',
 'nationality_code',
 'height',
 'weight')

In [0]:
df_sort.select('occupation').display()

In [0]:
display(df_final)

In [0]:
from pyspark.sql.functions import col

# Count athletes per nationality, sort descending, take top 10
df_top10 = df_final.groupBy("nationality") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(10)

df_top10.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

df_final.withColumn('cum_weight', F.sum('weight').over(Window.partitionBy('nationality').orderBy('height'))).display()

In [0]:
df_final.withColumn('cum_weight', F.sum('weight').over(Window.partitionBy('nationality').orderBy('height').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)))\
    .display()

In [0]:
df_final.createOrReplaceTempView("athletes")

In [0]:
df_new = spark.sql("""
SELECT 
    nationality,
    height,
    weight,
    SUM(weight) OVER (
        PARTITION BY nationality
        ORDER BY height
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS total_weight
FROM athletes
""")

In [0]:
df_new.display()

In [0]:
df_final.write.format("delta")\
    .mode("overwrite")\
        .option("path", "abfss://silver@xc797demo.dfs.core.windows.net/athletes")\
            .saveAsTable("olympic.silver.athletes")